- hidden-size는 **적당한** 값을 설정합니다.
- 수치해석적 미분을 통해서도 학습가능?
- 정확도 측정할 떄에도 배치사이즈로 나누네. 
  - 근데 그럼 만약 하나의 엄청 큰 오류사진, 여러 개의 약간 오류사진. 모두 data-driven하게 가중치 업데이트 시킨다면, 전자는 사실 조금만 되어야하구, 후자는 많이 되어야하지 않나? ~~특히 데이터가 표준화되어있지 않다면?~~




In [1]:
import os, sys
sys.path.append('./official_github/')

import numpy as np

from dataset.mnist import load_mnist
from my_functions import *

In [2]:
def function_2(x):
    return 2*x[0] + 3*x[1]

x = np.array([2.0, 4.0])
numerical_gradient(function_2, x)

2.0
True
16.0
1.9999999999999998
False
16.0
4.0
True
16.0
4.0
True
16.0


array([0., 0.])

2.0
[ True False]
16.0
1.9999999999999998
[False False]
16.0
4.0
[False  True]
16.0
4.0
[False  True]
16.0

In [13]:
class TwoLayerNet():
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01) -> None:
        
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)
        
    
    def predict(self, x):
        a1 = sigmoid(x @ self.params['W1'] + self.params['b1'])
        a2 = sigmoid(a1 @ self.params['W2'] + self.params['b2'])
        out = softmax(a2)
        return out
    
    def loss(self, x, t):
        y = self.predict(x)
        return cross_entropy_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        accuracy = np.sum(y == t) / x.shape[0]
        
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)
        
        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])
        
        return grads
    

In [14]:
(x_train, t_train), (x_test, t_test) = load_mnist(one_hot_label=False)

x_train.shape
# x_train[59_999]

(60000, 784)

In [15]:
train_loss_list = []

epoch = 1
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

network = TwoLayerNet(784, 100, 10)


def data_loader(data: np.ndarray, batch_size=100) -> np.ndarray:
    batch_indexes= np.arange(len(data))
    # np.random.shuffle(batch_indexes)
    while batch_indexes.any():
        batch = batch_indexes[:batch_size]
        # print(batch)
        batch_indexes = batch_indexes[batch_size:]
        yield data[batch, ...]

train_loss_list = []
for epoch_idx in range(1, epoch+1):
    print(f"===== {epoch_idx} epoch started =====")
    x_train_loader = data_loader(x_train)
    t_train_loader = data_loader(t_train)
    
    for x_batch, t_batch in zip(x_train_loader, t_train_loader):
        print(x_batch.shape, t_batch.shape)
        print('predict x batch shape', network.predict(x_batch).shape)
        grad = network.numerical_gradient(x_batch, t_batch)
        
        for key in network.params.keys():
            network.params[key] -= learning_rate * grad[key]
    
    loss = network.loss(x_batch, t_batch)
    print('loss:', loss)
    train_loss_list.append(loss)
    
    

===== 1 epoch started =====
(100, 784) (100,)
predict x batch shape (100, 10)


KeyboardInterrupt: 